# Fáza 3: Downstream Face Recognition — Full Pipeline

This notebook runs the **entire Phase 3 downstream experiment** on Google Colab with GPU:

1. **Setup** — Clone repo, install dependencies, authenticate W&B
2. **Data** — Download WebFace4M, prepare datasets for E1/E2/E3
3. **Style Transfer** — Generate newspaper-style augmented images with trained CUT
4. **Training** — Train E1 (baseline), E2 (augmented), E3 (mixed)
5. **Evaluation** — Rank-1, FAR/FRR, TAR@FAR on held-out test set
6. **Comparison** — Side-by-side results table + W&B dashboard

> **GPU Required**: Go to `Runtime → Change runtime type → T4 GPU` (or A100 if available).

---
## 0. Configuration

Edit these values before running:

In [ ]:
# ============================================================
# CONFIGURATION — EDIT THESE VALUES
# ============================================================

# W&B settings
WANDB_ENTITY = "knn-proj"                    
WANDB_PROJECT = "downstream-face-rec"         # W&B project name

# Git
REPO_URL = "https://github.com/jetoadka/knn-proj.git"
BRANCH = "feature/downstream"

# Data — how many WebFace4M shards to use (each has ~53k images)
# More shards = better results but longer training
# 2 shards (~106k images) is good for a quick test
# 10 shards (~530k images) is recommended for real experiments
NUM_WEBFACE_SHARDS = 2

# Training
BACKBONE = "convnext_atto"       # lightweight backbone, fast training
LOSS = "cosface"                  # cosface | arcface | adaface
EMBEDDING_DIM = 512
BATCH_SIZE = 64                   # increase to 128-256 on A100
EPOCHS = 30                       # reduce to 5-10 for quick test
LR = 1e-3
WEIGHT_DECAY = 1e-1
EVAL_INTERVAL = 5                 # evaluate every N epochs

# Use Google Drive for persistent storage (recommended)
USE_GOOGLE_DRIVE = True
DRIVE_BASE = "/content/drive/MyDrive/knn-proj"  # path in Google Drive

print("Configuration loaded")

---
## 1. Setup Environment

In [ ]:
# Check GPU availability
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("No GPU detected! Go to Runtime → Change runtime type → GPU")
    raise RuntimeError("GPU required")

In [ ]:
# Mount Google Drive (for persistent data/checkpoints across sessions)
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.makedirs(DRIVE_BASE, exist_ok=True)
    print(f"Google Drive mounted, base dir: {DRIVE_BASE}")
else:
    print("Google Drive not mounted — data will be lost on disconnect")

In [ ]:
# Clone repository
import os

WORK_DIR = "/content/knn-proj"

if not os.path.exists(WORK_DIR):
    !git clone -b {BRANCH} {REPO_URL} {WORK_DIR}
    print(f"Cloned {BRANCH} to {WORK_DIR}")
else:
    !cd {WORK_DIR} && git pull
    print(f"Repo already exists, pulled latest changes")

os.chdir(WORK_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!pip install -q pytest scikit-learn
print("\nDependencies installed")

In [ ]:
# Authenticate W&B
import wandb

wandb.login()
print(f"Logged in to W&B")
print(f"   Entity: {WANDB_ENTITY}")
print(f"   Project: {WANDB_PROJECT}")
print(f"   Dashboard: https://wandb.ai/{WANDB_ENTITY}/{WANDB_PROJECT}")

In [ ]:
# Quick sanity check — run unit tests
!python -m pytest tests/ -v --tb=short 2>&1 | tail -20

---
## 2. Download & Prepare Data

This step downloads WebFace4M shards from HuggingFace and extracts them.
If using Google Drive, data is cached across sessions.

In [ ]:
import os
from pathlib import Path

# Set up data directories
if USE_GOOGLE_DRIVE:
    DATA_DIR = Path(DRIVE_BASE) / "data"
    CKPT_DIR = Path(DRIVE_BASE) / "checkpoints"
else:
    DATA_DIR = Path("/content/knn-proj/data")
    CKPT_DIR = Path("/content/knn-proj/checkpoints")

# Create symlinks so scripts can find data at standard paths
data_link = Path("/content/knn-proj/data")
ckpt_link = Path("/content/knn-proj/checkpoints")

DATA_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

if not data_link.exists():
    os.symlink(str(DATA_DIR), str(data_link))
if not ckpt_link.exists():
    os.symlink(str(CKPT_DIR), str(ckpt_link))

print(f"Data dir: {DATA_DIR}")
print(f"Checkpoint dir: {CKPT_DIR}")

In [ ]:
# Download WebFace4M shards
WEBFACE_DIR = DATA_DIR / "webface4m"

# Check if already downloaded
existing_shards = list(WEBFACE_DIR.glob("*.tar.gz")) if WEBFACE_DIR.exists() else []
if len(existing_shards) >= NUM_WEBFACE_SHARDS:
    print(f"WebFace4M already downloaded ({len(existing_shards)} shards)")
else:
    print(f"Downloading {NUM_WEBFACE_SHARDS} WebFace4M shards (~{NUM_WEBFACE_SHARDS * 80} MB)...")
    !python -m src.data_prep.download_webface4m \
        --output-dir {WEBFACE_DIR} \
        --num-shards {NUM_WEBFACE_SHARDS} \
        --verify
    print(" WebFace4M download complete")

In [ ]:
# Extract WebFace4M shards to flat image directory
import tarfile
import io
from PIL import Image
from tqdm.auto import tqdm

WEBFACE_FLAT = DATA_DIR / "webface4m_flat"

if WEBFACE_FLAT.exists() and len(list(WEBFACE_FLAT.glob("*.jpg"))) > 100:
    n_existing = len(list(WEBFACE_FLAT.glob("*.jpg")))
    print(f" WebFace4M already extracted ({n_existing:,} images)")
else:
    WEBFACE_FLAT.mkdir(parents=True, exist_ok=True)
    shard_files = sorted(WEBFACE_DIR.glob("*.tar.gz"))
    print(f"Extracting {len(shard_files)} shards to {WEBFACE_FLAT}...")

    total_images = 0
    class_labels = {}  # {key: cls}

    for shard_path in shard_files:
        print(f"  Extracting {shard_path.name}...")
        with tarfile.open(shard_path, 'r:gz') as tar:
            members = tar.getmembers()
            # Group by key (each sample has key.jpg + key.cls)
            jpgs = {m.name.replace('.jpg', ''): m for m in members if m.name.endswith('.jpg')}
            clss = {m.name.replace('.cls', ''): m for m in members if m.name.endswith('.cls')}

            for key in tqdm(jpgs, desc=shard_path.stem, leave=False):
                if key in clss:
                    # Read class label
                    cls_f = tar.extractfile(clss[key])
                    if cls_f:
                        cls_id = cls_f.read().decode().strip()
                    else:
                        continue

                    # Extract image using identity as filename prefix
                    # Format: IDENTITY_SEQNUM.jpg (so our dataset loader can parse identity)
                    key_basename = key.split('/')[-1] if '/' in key else key
                    out_name = f"{cls_id}_{key_basename}.jpg"
                    out_path = WEBFACE_FLAT / out_name

                    if not out_path.exists():
                        jpg_f = tar.extractfile(jpgs[key])
                        if jpg_f:
                            out_path.write_bytes(jpg_f.read())
                            total_images += 1

    print(f"\n Extracted {total_images:,} images to {WEBFACE_FLAT}")

In [ ]:
# Verify data
import subprocess

webface_count = int(subprocess.check_output(f"find {WEBFACE_FLAT} -name '*.jpg' | wc -l", shell=True).strip())
sample_gator_test = Path("sample_data/people_gator/aligned_112/test")
gator_test_count = len(list(sample_gator_test.rglob("*.jpg"))) + len(list(sample_gator_test.rglob("*.png")))

print(f"Data Summary:")
print(f"   WebFace4M (flat): {webface_count:,} images")
print(f"   people_gator test: {gator_test_count} images")
print(f"   Sample data:  available in sample_data/")

---
## 3. Generate Newspaper-Style Augmented Data (CUT)

Run the trained CUT generator on WebFace4M to produce newspaper-style images.

> **Requires a trained CUT checkpoint!** If you haven't trained CUT yet,
> either:
> - Upload your trained checkpoint to Google Drive
> - Or skip this step and run E1 (baseline) only

In [ ]:
# Check for CUT checkpoint
CUT_CKPT_DIR = Path("src/style_transfer/cut_model/checkpoints/exp_combined")
DRIVE_CUT_CKPT = Path(DRIVE_BASE) / "cut_checkpoints" if USE_GOOGLE_DRIVE else None

# Try to find checkpoint
cut_checkpoint_found = False

if CUT_CKPT_DIR.exists() and list(CUT_CKPT_DIR.glob("*_net_G.pth")):
    cut_checkpoint_found = True
    print(f"CUT checkpoint found at {CUT_CKPT_DIR}")
elif DRIVE_CUT_CKPT and DRIVE_CUT_CKPT.exists():
    # Copy from Google Drive
    CUT_CKPT_DIR.mkdir(parents=True, exist_ok=True)
    !cp -v {DRIVE_CUT_CKPT}/*_net_G.pth {CUT_CKPT_DIR}/
    cut_checkpoint_found = True
    print(f"CUT checkpoint copied from Google Drive")
else:
    print("No CUT checkpoint found!")
    print(f"   Expected at: {CUT_CKPT_DIR}")
    if USE_GOOGLE_DRIVE:
        print(f"   Or upload to: {DRIVE_CUT_CKPT}")
    print("\n   You can still run E1 (baseline-clean) without CUT.")
    print("   To generate augmented data, upload latest_net_G.pth to the checkpoint dir.")

In [ ]:
# Generate augmented newspaper-style data
AUGMENTED_DIR = DATA_DIR / "augmented_newspaper"

if not cut_checkpoint_found:
    print(" Skipping augmented data generation (no CUT checkpoint)")
    print("   E2 and E3 experiments will not be available.")
elif AUGMENTED_DIR.exists() and len(list(AUGMENTED_DIR.glob("*.jpg"))) > 100:
    n_aug = len(list(AUGMENTED_DIR.glob("*.jpg")))
    print(f" Augmented data already generated ({n_aug:,} images)")
else:
    print(f" Generating newspaper-style images from WebFace4M...")
    print(f"   Input:  {WEBFACE_FLAT}")
    print(f"   Output: {AUGMENTED_DIR}")
    print(f"   This may take 30-60 minutes on T4 GPU...\n")

    !python -m src.downstream.generate_augmented_data \
        --input-dir {WEBFACE_FLAT} \
        --output-dir {AUGMENTED_DIR} \
        --checkpoint-dir {CUT_CKPT_DIR} \
        --batch-size 32 \
        --device cuda

    print(f"\n Augmented data generation complete")

---
## 4. Prepare Training Datasets

Create the combined datasets for each experiment:
- **E1**: WebFace4M only (baseline)
- **E2**: WebFace4M + CUT-generated newspaper-style
- **E3**: WebFace4M + CUT-generated + real newspaper (people_gator train)

In [ ]:
from pathlib import Path

DOWNSTREAM_DIR = DATA_DIR / "downstream"
AUGMENTED_DIR = DATA_DIR / "augmented_newspaper"

# ---- E1: Baseline (clean only) ----
E1_DIR = DOWNSTREAM_DIR / "E1_baseline" / "train"
if E1_DIR.exists() and any(E1_DIR.iterdir()):
    print(f" E1 dataset already prepared")
else:
    print(" Preparing E1 dataset (baseline — clean only)...")
    !python -m src.downstream.prepare_timm_dataset \
        --sources {WEBFACE_FLAT} \
        --output-dir {E1_DIR} \
        --mode train --symlink

# ---- E2: Augmented (clean + CUT-generated) ----
if cut_checkpoint_found:
    E2_DIR = DOWNSTREAM_DIR / "E2_augmented" / "train"
    if E2_DIR.exists() and any(E2_DIR.iterdir()):
        print(f" E2 dataset already prepared")
    else:
        print(" Preparing E2 dataset (clean + augmented)...")
        !python -m src.downstream.prepare_timm_dataset \
            --sources {WEBFACE_FLAT} {AUGMENTED_DIR} \
            --output-dir {E2_DIR} \
            --mode train --symlink

    # ---- E3: Mixed (clean + CUT-generated + real newspaper) ----
    E3_DIR = DOWNSTREAM_DIR / "E3_mixed" / "train"
    GATOR_TRAIN = Path("sample_data/people_gator/aligned_112/train")
    if E3_DIR.exists() and any(E3_DIR.iterdir()):
        print(f" E3 dataset already prepared")
    else:
        print(" Preparing E3 dataset (clean + augmented + real newspaper)...")
        !python -m src.downstream.prepare_timm_dataset \
            --sources {WEBFACE_FLAT} {AUGMENTED_DIR} {GATOR_TRAIN} \
            --output-dir {E3_DIR} \
            --mode train --symlink
else:
    print(" Skipping E2/E3 (no CUT checkpoint, no augmented data)")

print("\n Dataset preparation complete")

In [ ]:
# Prepare evaluation data
EVAL_DIR = DOWNSTREAM_DIR / "eval_test"
GATOR_TEST = Path("sample_data/people_gator/aligned_112/test")

if EVAL_DIR.exists() and (EVAL_DIR / "pairs.json").exists():
    print(f" Evaluation data already prepared")
else:
    print(" Preparing evaluation data (people_gator test)...")
    !python -m src.downstream.prepare_timm_dataset \
        --sources {GATOR_TEST} \
        --output-dir {EVAL_DIR} \
        --mode eval

---
## 5. Training

Train each experiment with W&B logging. Each run logs to
`wandb.ai/{ENTITY}/{PROJECT}` for live monitoring.

> 💡 You can monitor training in real-time at your W&B dashboard!

In [ ]:
# Helper function to run training for an experiment
import subprocess, sys

def train_experiment(
    run_name: str,
    train_dir: str,
    val_dir: str = "sample_data/people_gator/aligned_112/dev",
):
    """Train a face recognition experiment with W&B logging."""
    cmd = [
        sys.executable, "-m", "src.downstream.train_downstream",
        "--train-dir", str(train_dir),
        "--val-dir", str(val_dir),
        "--backbone", BACKBONE,
        "--loss", LOSS,
        "--embedding-dim", str(EMBEDDING_DIM),
        "--batch-size", str(BATCH_SIZE),
        "--epochs", str(EPOCHS),
        "--lr", str(LR),
        "--weight-decay", str(WEIGHT_DECAY),
        "--eval-interval", str(EVAL_INTERVAL),
        "--run-name", run_name,
        "--wandb-mode", "online",
        "--save-dir", str(CKPT_DIR),
        "--device", "cuda",
        "--num-workers", "2",
    ]

    print(f"\n{'='*60}")
    print(f" Starting: {run_name}")
    print(f"   Train dir: {train_dir}")
    print(f"   Val dir:   {val_dir}")
    print(f"   W&B:       https://wandb.ai/{WANDB_ENTITY}/{WANDB_PROJECT}")
    print(f"{'='*60}\n")

    # Patch the entity/project in the environment
    import os
    os.environ['WANDB_ENTITY'] = WANDB_ENTITY
    os.environ['WANDB_PROJECT'] = WANDB_PROJECT

    result = subprocess.run(cmd, env={**os.environ})
    if result.returncode == 0:
        print(f"\n {run_name} complete!")
    else:
        print(f"\n {run_name} failed with code {result.returncode}")
    return result.returncode

In [ ]:
# Patch train_downstream.py to use our configured entity/project
# (The script defaults to entity="knn-proj" — we override via env vars)

import os
os.environ['WANDB_ENTITY'] = WANDB_ENTITY
os.environ['WANDB_PROJECT'] = WANDB_PROJECT
print(f" W&B configured: {WANDB_ENTITY}/{WANDB_PROJECT}")

### 5.1 E1: Baseline (Clean Only)

In [ ]:
E1_TRAIN = DOWNSTREAM_DIR / "E1_baseline" / "train"
train_experiment("E1-baseline-clean", E1_TRAIN)

### 5.2 E2: Augmented (Clean + CUT-Generated)

> Skip this cell if you don't have a CUT checkpoint.

In [ ]:
if cut_checkpoint_found:
    E2_TRAIN = DOWNSTREAM_DIR / "E2_augmented" / "train"
    train_experiment("E2-augmented-newspaper", E2_TRAIN)
else:
    print(" Skipping E2 (no CUT checkpoint / no augmented data)")

### 5.3 E3: Mixed (Clean + CUT-Generated + Real Newspaper)

> Skip this cell if you don't have a CUT checkpoint.

In [ ]:
if cut_checkpoint_found:
    E3_TRAIN = DOWNSTREAM_DIR / "E3_mixed" / "train"
    train_experiment("E3-augmented-mixed", E3_TRAIN)
else:
    print(" Skipping E3 (no CUT checkpoint / no augmented data)")

---
## 6. Evaluation

Evaluate all trained models on the held-out test set with full metrics:
- **Rank-1 / Rank-5 accuracy**
- **TAR @ FAR = 1e-4**
- **10-fold verification accuracy**

In [ ]:
from pathlib import Path

# Discover trained checkpoints
EVAL_TEST_DIR = Path("sample_data/people_gator/aligned_112/test")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

experiments = [
    ("E1-baseline-clean", CKPT_DIR / "E1-baseline-clean" / "best_model.pth"),
]

if cut_checkpoint_found:
    experiments.extend([
        ("E2-augmented-newspaper", CKPT_DIR / "E2-augmented-newspaper" / "best_model.pth"),
        ("E3-augmented-mixed", CKPT_DIR / "E3-augmented-mixed" / "best_model.pth"),
    ])

# Filter to existing checkpoints
valid_experiments = [(name, path) for name, path in experiments if path.exists()]

if not valid_experiments:
    # Try fallback: find any .pth files
    print(" No best_model.pth found. Searching for any checkpoint...")
    for name, path in experiments:
        parent = path.parent
        if parent.exists():
            pth_files = sorted(parent.glob("epoch_*.pth"))
            if pth_files:
                valid_experiments.append((name, pth_files[-1]))
                print(f"  Found: {pth_files[-1]}")

print(f"\n Experiments to evaluate: {len(valid_experiments)}")
for name, path in valid_experiments:
    print(f"   {name}: {path}")

In [ ]:
# Run evaluation for all experiments
if valid_experiments:
    checkpoint_args = []
    name_args = []
    for name, path in valid_experiments:
        checkpoint_args.extend(["--checkpoint", str(path)])
        name_args.extend(["--experiment-name", name])

    # Build the full command
    ckpt_str = " ".join(str(p) for _, p in valid_experiments)
    name_str = " ".join(n for n, _ in valid_experiments)

    !python -m src.downstream.evaluate \
        --checkpoint {ckpt_str} \
        --test-dir {EVAL_TEST_DIR} \
        --backbone {BACKBONE} \
        --embedding-dim {EMBEDDING_DIM} \
        --experiment-name {name_str} \
        --device cuda \
        --wandb-mode online \
        --output results/final_comparison.json
else:
    print(" No checkpoints found to evaluate")

In [ ]:
# Display results
import json

results_file = Path("results/final_comparison.json")
if results_file.exists():
    with open(results_file) as f:
        results = json.load(f)

    print(f"\n{'='*80}")
    print(f" FINAL RESULTS")
    print(f"{'='*80}")
    print(f"{'Experiment':<30} {'Rank-1':>8} {'Rank-5':>8} {'TAR@1e-4':>10} {'10-fold':>8}")
    print("-" * 70)

    for r in results:
        name = r.get('experiment', '?')
        r1 = f"{r['rank1_accuracy']:.4f}" if r.get('rank1_accuracy') is not None else '—'
        r5 = f"{r['rank5_accuracy']:.4f}" if r.get('rank5_accuracy') is not None else '—'
        tar = f"{r['tar_at_far_1e4']:.4f}" if r.get('tar_at_far_1e4') is not None else '—'
        kf = f"{r['kfold_verification_accuracy']:.4f}" if r.get('kfold_verification_accuracy') is not None else '—'
        print(f"{name:<30} {r1:>8} {r5:>8} {tar:>10} {kf:>8}")

    print("-" * 70)
    print(f"\n Full results saved to: {results_file}")
else:
    print(" No results file found")

---
## 7. W&B Comparison Dashboard

View all experiments side-by-side on your W&B dashboard.

In [ ]:
# Generate comparison report
!python -m src.evaluation.compare_experiments \
    --entity {WANDB_ENTITY} \
    --project {WANDB_PROJECT} \
    --markdown results/comparison_report.md \
    --output results/comparison_data.json

print(f"\n🔗 View dashboard: https://wandb.ai/{WANDB_ENTITY}/{WANDB_PROJECT}")

In [ ]:
# Display the markdown report inline
from IPython.display import Markdown, display

report_file = Path("results/comparison_report.md")
if report_file.exists():
    display(Markdown(report_file.read_text()))
else:
    print("No comparison report generated yet (need completed W&B runs)")

---
## 8. Save Results to Google Drive

Copy checkpoints and results to Google Drive for persistent storage.

In [ ]:
if USE_GOOGLE_DRIVE:
    import shutil

    # Copy results
    drive_results = Path(DRIVE_BASE) / "results"
    drive_results.mkdir(parents=True, exist_ok=True)

    for f in Path("results").glob("*"):
        shutil.copy2(f, drive_results / f.name)
        print(f"  📄 Saved {f.name} to Google Drive")

    print(f"\n✅ All results saved to {drive_results}")
    print(f"   Checkpoints at: {CKPT_DIR}")
else:
    print("⚠️ Google Drive not mounted — download results manually before disconnecting!")
    from google.colab import files
    if Path("results/final_comparison.json").exists():
        files.download("results/final_comparison.json")

---
## 📝 Summary

| Step | Status |
|------|--------|
| Environment setup | ✅ |
| Data download & extraction | ✅ |
| CUT augmented data generation | ✅ (if checkpoint available) |
| E1 training (baseline) | ✅ |
| E2 training (augmented) | ✅ (if CUT data available) |
| E3 training (mixed) | ✅ (if CUT data available) |
| Evaluation on test set | ✅ |
| W&B comparison dashboard | ✅ |
| Results saved to Drive | ✅ |

### Next steps:
1. Visit your [W&B dashboard](https://wandb.ai) to explore the results
2. Use the comparison table to decide which augmentation strategy works best
3. Include the results in your project report